In [ ]:
import pandas as pd

from src.extract.save_dir import saved_data_dir

In [ ]:
#supply data
generated_time=pd.date_range(start='2016-01-01', end='2026-01-01',freq='D')
generated_time=generated_time.strftime('%Y-%m-%d')
odd=generated_time[::2]
even=generated_time[1::2]
combined={}
for o,e in zip(odd,even):
    combined[o]=e

dfs=[]
for o,e in combined.items():
    url=f'https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH?settlementDateFrom={o}&settlementDateTo={e}&format=csv'
    df=pd.read_csv(url)
    if len(df)>0:
        dfs.append(df)
    else:
        print(f'{o}-{e} no data')

In [ ]:
from src.extract.save_dir import saved_data_dir

In [ ]:
supply=pd.concat(dfs)
saved_data_dir('supply_original.csv',supply,'../data/raw/')

In [ ]:
supply=pd.read_csv('../data/raw/supply_original.csv')
supply['StartTime']=pd.to_datetime(supply['StartTime'],utc=False).dt.tz_localize(None)
print(supply.tail())
print(supply.shape)
print(supply.columns)
print(supply.head())

In [ ]:
supply=supply.drop(columns='Dataset')
supply=supply.drop(columns='SettlementDate')
supply=supply.drop(columns='PublishTime')
supply=supply.drop(columns='SettlementPeriod')
print(supply.head())

In [ ]:
#dataset form adjust
pivot=pd.pivot_table(
    index='StartTime',
    columns='FuelType',
    values='Generation',
    data=supply,
    aggfunc='mean',
    fill_value=0
).reset_index()
pivot.columns.name=None
print(pivot.head())
print(pivot.isna().sum())
print(pivot.duplicated().sum())
print(pivot.columns)

In [ ]:
from src.transform.half_hour_to_hour import hh_to_hour

In [ ]:
x_list=pivot.columns.tolist()
x_list.remove('StartTime')
hh_to_hour('StartTime',pivot,x_list,'../data/interim/hourly_supply_pivot.csv')